# CasDsl — a categorically organized CAS in Lean 4

Every cell in this notebook is written in CasDsl, a computer-algebra
surface [elaborated](https://lean-lang.org/doc/reference/latest/Elaboration-and-Compilation/)
by Lean 4 — type-checked by the same machinery that verifies proofs,
though no cell creates a theorem; an assertion is only as strong as the
backend that answered it. You need no Lean to read the cells; the
surface is ordinary mathematical notation.

**The organizing philosophy: everything is a category.** A value is
instantiated into its categories, and the category decides which methods
it carries: `2 + 2i` is a complex number, so it has `.re()`, `.im()`,
`|·|`; a square matrix has `.det()`; a polynomial has `.roots()` and
`.factor()`. This is what makes Sage a "catalogue of algorithms" — with
the categories actually defined, and methods that travel along
inclusions by functorial transport: ℤ ≤ ℚ ≤ ℝ ≤ ℂ, Euclidean ≤ UFD, so
an operation is declared once, where it first makes sense, and inherited
everywhere below.

**Backend-blind syntax.** No expression ever names Sage, GAP, or an
algorithm: you ask for the mathematical operation, and a backend answers
— Sage today, with Python, GAP, Singular, Julia, and native binaries in
the pool. Backends are interchangeable, and the surface never sees them.

In an ordinary Jupyter notebook, the kernel is a REPL: context persists
between cell runs, and every run — in any order, any number of times —
appends to that context. A cell's output records the context at the
moment it ran, which can diverge from the notebook as it reads on
screen: re-run a cell or run cells out of order, and outputs no longer
line up with the code. This notebook runs on a kernel that excludes that
failure: the notebook is a **document, not a transcript**. Each cell's
output is the result of elaborating the visible prefix of the notebook
through that cell, so outputs always correspond to the code shown above
them. Editing a cell re-runs it and re-establishes everything below it —
visibly, marked with `↻`, never silently. Unchanged cells are cached, so
editing cell 5 does not re-run cell 1; an upstream failure stops the run
with an `UpstreamError` rather than leaving the notebook half-applied.
In an ordinary notebook, editing is free and outputs drift from the
code; here editing makes the notebook recompute — surprising until you
see the point: what you read is what was computed.


## 1 · Assertions

`assert` takes a **proposition** — a decidable statement. The primitive
proposition is equality: `2 + 3 = 5` claims the two sides denote the
same element of the ambient domain. The other relations — `≠`, `∈`,
`∉`, `⊆` — form propositions too, and `and` conjoins several into one
assertion.

A proposition is three-valued: `true`, `false`, or `unknown`. `assert P`
demands `true` — that commits the cell. `false` fails it, rolling the
notebook state back to the last committed cell. `unknown` can occur when
the two sides could not be compared in the ambient domain; it is a
failure too, distinct from `false`. A fourth outcome, `error`, is not a
truth value but a machine failure: no implementation exists or a backend
died, and the system says so.

Nothing is proved: no Lean theorem is generated. `assert 2 + 3 = 5`
holds because a backend computed `2 + 3` and got `5` — the assertion's
whole content is what the backend reported.


### The reserved surface

The grammar's reserved words and symbols, with what they do:

| token | meaning |
|---|---|
| `let x := e in D` | bind `x` to `e`, ascribed to a domain or category `D` (checked) |
| `let p(x) := e in D` | define `p` with indeterminate `x` — e.g. `ℤ[x]` |
| `assert P` | decide `P` — three truth values, §1 |
| `map e to D` | transport `e` along the canonical map into `D` |
| `x ↦ e` | a function; `|->` is the ASCII spelling |
| `→`, `->` | the domain arrow of a function type |
| `=`, `is`, `≠`, `∈`, `in`, `∉`, `⊆` | assertion relations (`\in`, `\leq` are the backslash spellings) |
| `√` | a chosen branch of the multivalued square root: `√2` is the non-negative root, `√−1` is the primitive fourth root of unity, `i` |
| `i`, `e` | never shadowable: the imaginary unit (a primitive fourth root of unity), Euler's number (`e^t = exp t`) |
| `d`, `π` | shadowable constants: the differential, pi |
| `dx` | the differential atom — `∫ f dx`, `d(f) = (6x + 1) dx` |
| `O(ε)` | approximation tolerance: `map √2 to ℝ/O(1/10^{10})` |
| `…` | set continuation: `{0, 2, 4, …}` |
| `#capabilities` | inspect the dispatch: also `#explain_route e`, `#capability_gaps` |

Only the first column's true tokens — `dx`, `Spec`, `lim_`, `map`, `to`,
`is`, and the constants `i`, `e` — are words no binding may shadow; the
other spellings (`and`, `O`, `dt`) are ordinary identifiers elsewhere in
the grammar.


In [1]:
assert 2 + 3 = 5

Starting Lean worker (/tmp/journey-proof-70d28d7/qualification/lean-cas-dsl)…
1:0: ✓ 2 + 3 = 5


The `in ℤ/5` suffix changes the ring the assertion is evaluated in.
`2 + 3` is 5 in ℤ, but 5 ≡ 0 in ℤ/5, so the assertion holds.

In [2]:
assert 2 + 3 = 0 in ℤ/5

1:0: ✓ 2 + 3 = 0 in ℤ/5


`and` chains decide conjunct by conjunct: one commit when every conjunct is true, and a refusal names the conjunct that failed. `is` is just `=` in the SPEC's spelling.

In [3]:
assert 2 + 3 = 5 and 4 ∉ {1, 2, 3}

1:0: ✓ 2 + 3 = 5 and 4 ∉ {1, 2, 3}


In [4]:
assert 2 + 3 is 5

1:0: ✓ 2 + 3 = 5


The reserved surface bites in bindings too: `e` is Euler's constant and `i` the imaginary unit, so no binding may shadow either — both mean the same thing in every session state. The refusals below are the point: they fail loudly instead of silently rebinding.

In [5]:
let e := 5 in ℤ

LeanError: `e` is Euler's constant, a reserved symbol of this surface (owner ruling 2026-07-31): `e^t` means exp(t), `exp(x)` spells the same function, and no binding may shadow it

In [ ]:
let i := 3 in ℤ

## 2 · Factorization

`factor` is a method on elements of a unique factorization domain (UFD):
the category is named for the theorem that holds there — factorization
into irreducibles, unique up to units and order. An integer receives it
because ℤ is Euclidean and every Euclidean domain is a UFD: the method
arrives along the inclusion, not by per-ring forwarding code. Sage
performs the computation, but `n.factor()` never names it.


In [ ]:
let n := 360 in ℤ

In [ ]:
n.factor()

Routes are inspectable: `#explain_route` names the backend and the route a computation took, textually. And a literal numeral is not a receiver — `360.factor()` is a tokenizer casualty (the lexer reads `360.` as a decimal before any production sees it) — so the receiver is parenthesized, and the method resolves on the receiver's VALUE.

In [ ]:
#explain_route n.factor()

In [ ]:
(360).factor()

In [ ]:
assert (84).gcd(30) = 6

`gcd` works the same way — a method on UFD elements, where a greatest
common divisor is unique up to units, and ℤ inherits it through the same
inclusion.


In [ ]:
gcd(84, 30) = 6

## 3 · Polynomials

`roots()` and `factor()` are answered in the ring a polynomial is bound
to: the same $p$ can live in $\mathbb{Z}[x]$, $\mathbb{Q}[x]$, or
$\mathbb{C}[x]$, and the answer changes with the ring.

The `(x)` in `let p(x) := … in ℤ[x]` declares `x` as the indeterminate.

In [ ]:
let p(x) := x^5 - x^4 - x^3 + x^2 - 2x + 2 in ℤ[x]

Evaluation is substitution. Is $1$ a root?

In [ ]:
assert p(1) = 0

`p.roots()` asks for the roots **in the coefficient ring** — here, an
integer. How many roots does $p$ have in ℤ?

In [ ]:
p.roots()

`p.factor()` asks the companion question: the factorization into
irreducibles, with multiplicity. The irreducible quadratic factors are
precisely the roots the ring cannot see.

In [ ]:
p.factor()

Move $p$ to $\mathbb{Q}[x]$ along the canonical inclusion
$\mathbb{Z} \subseteq \mathbb{Q}$, coefficient by coefficient.

For a monic integer polynomial, Gauss: rational roots are integers, and
irreducibility over ℤ ⟺ over ℚ. ℚ is not a splitting field for either
quadratic, so the quadratics stay irreducible.


In [ ]:
let pQ := map p to ℚ[x]

In [ ]:
pQ.roots()

In [ ]:
pQ.factor()

Map to $\mathbb{C}[x]$. ℂ is algebraically closed: $p$ splits into
linear factors, and the root set becomes the full census. The census is
taken in the ring asked for, and no further.


In [ ]:
let pC := map pQ to ℂ[x]

In [ ]:
pC.roots()

In [ ]:
pC.factor()

Definitions use `:=` — SPEC's opening sentence. `NAME := expr` is a command, sugar for `let` through the same elaborator; SPEC's own §Polynomials line is `q := map p to ℂ[x]`. Maps along canonical inclusions compose freely with the polynomial commands.

In [ ]:
let pb(x) := x^3 - 2x + 1 in ℤ[x]

In [ ]:
qb := map pb to ℂ[x]

In [ ]:
assert qb(1) = 0

The root set by type: $1$ (the integer root); $\pm\sqrt{2}$ — a pair
that travels together, the two roots of the irreducible $x^2 - 2$; and
$\pm i$ — the complex-conjugate pair. Three irreducibles over ℤ become
five linear factors over ℂ. Nothing about $p$ changed — the ring did.

In [ ]:
assert |pC.roots()| = 5
assert ∑_{a ∈ pC.roots()} a = 1
assert ∏_{a ∈ pC.roots()} a = -2

## 4 · Exact algebraic numbers

CasDsl works with exact algebraic numbers — never decimals unless you
explicitly request an approximation. The ⊆-chain
$\mathbb{N} \subseteq \mathbb{Z} \subseteq \mathbb{Q} \subseteq \mathbb{R} \subseteq \mathbb{C}$
is built in: membership and transport are automatic.


In [ ]:
let z := 2 + 2i in ℂ

The absolute value is an exact algebraic operation: `|·|` returns the
surd — simplification carried out exactly, never approximated as a
decimal.

In [ ]:
assert |z| = 2√2

Numerical approximation is an operation **on** an exact value, not a
replacement for it. `map √2 to ℝ/O(1/10^{10})` asks for $\sqrt{2}$
approximated to within $10^{-10}$ in ℝ. The tolerance is a request, not a
quotient — the underlying value remains exact.

In [ ]:
map √2 to ℝ/O(1/10^{10})

Membership is a decided proposition, and the exact values live where they belong: `√2 ∈ ℝ` and `2 + 2i ∈ ℂ` are SPEC's own lines — no decimal anywhere.

In [ ]:
assert √2 ∈ ℝ

In [ ]:
assert 2 + 2i ∈ ℂ

The number-system chain is decided by the registry of canonical maps: `ℤ ⊆ ℚ and ℚ ⊆ ℝ and ℝ ⊆ ℂ` is SPEC's own line, one commit. The visible consequence: an integer ascribes to ℝ or ℂ through a registered canonical map.

In [ ]:
assert ℤ ⊆ ℚ and ℚ ⊆ ℝ and ℝ ⊆ ℂ

In [ ]:
let rr := 3 in ℝ

In [ ]:
map 3 to ℂ

The bare-definition sugar carries its ascription tail: `zb := 2 + 2i in ℂ` checks the membership exactly as `let` would.

In [ ]:
zb := 2 + 2i in ℂ

In [ ]:
assert zb.re() = 2

## 5 · Linear algebra

Exact matrix arithmetic over ℚ. Matrix literals use **row-semicolon
syntax**: `[1, 2; 3, 4]` is a $2\times 2$ matrix whose rows are `[1, 2]`
and `[3, 4]`.

In [ ]:
let M := [1, 2; 3, 4] in Mat₂(ℚ)

The determinant is a category-owned method on square matrices, exact
over ℚ — no floating point.

In [ ]:
assert M.det() = -2

`M⁻¹` exists because $\det(M) \ne 0$, and over ℚ it is computed
exactly: rational entries, no decimals.

In [ ]:
M⁻¹

Rank and kernel are category-owned methods: `M` is invertible, so it kills nothing — `ker M = {0}`, and `0` is the zero of the ambient space ℚ².

In [ ]:
assert M.rank() = 2

In [ ]:
assert M.ker() = {0}

In [ ]:
M.ker()

`and` survives identifier conjuncts — every conjunct here ends in a NAME, the shape the juxtaposition parser is careful about.

In [ ]:
let v := (1, 0) in ℚ²

In [ ]:
let b := (1, 3) in ℚ²

In [ ]:
assert M v = b and v = v

Transport along a forgetful functor: `F` is the ℤ-module ℤ/4 in `SmallModules(ℤ)`, a subcategory of `Modules(ℤ)`, which is a subcategory of `Sets`. Module methods resolve directly — the annihilator needs no help — while cardinality is a Sets question that arrives through the functor, and `#explain_route` shows the path.

In [ ]:
let F := ℤ/4 in SmallModules(ℤ)

In [ ]:
F.annihilator()

In [ ]:
F.cardinality()

In [ ]:
#explain_route F.cardinality()

Bare `=` never inserts the functor: `F` is a module, `{0, 1, 2, 3}` is a set, and cross-category equality is trivially false — the refusal says so. The Sets question is one explicit call away, receiver transported:

In [ ]:
assert F = {0, 1, 2, 3}

In [ ]:
F.set_eq({0, 1, 2, 3})

## 6 · Calculus

CasDsl distinguishes the **universal differential** $d(f)$ — a 1-form —
from the **derivation** $(d/dx)(f)$ — a polynomial. They are different
types and not equal, even when their coefficients match.

The indefinite integral $\int f\,dx$ returns the **coset** of
antiderivatives — the set of all primitives, not a choice of constant.

In [ ]:
let f := x ↦ 3x² + x + 1 in ℚ[x]

`d` is the universal relative differential: `d(f)` is a 1-form, and the
`dx` is part of the value, not decoration.

In [ ]:
assert d(f) = (6x + 1) dx

`(d/dx)(f)` is the derivation: a plain polynomial. Note the absence of
`dx` — this is the coefficient of the differential, not the
differential itself.

In [ ]:
assert (d/dx)(f) = 6x + 1

The $+ \mathbb{Q}$ is the constant of integration, presented as a coset:
any rational constant can be added to a primitive and the result is
still a primitive. This is not a notational convention — the integral is
a set.


In [ ]:
∫ f dx

The kernel of the derivation is exactly the constant field:

In [ ]:
assert kernel(d/dx : ℚ[x] → ℚ[x]) = ℚ

The coset is usable, not just displayable: select the primitives with
`h(0) = 0` — there is exactly one — pick it out, and check it against
the differential and the derivation.

In [ ]:
let Fs := {h ∈ ∫ f dx | h(0) = 0} in 𝒫(ℚ[x])
assert Fs.cardinality() = 1
let F := Fs[0] in ℚ[x]
assert F(x) = x³ + (1/2)x² + x
assert d(F) = f dx

## 7 · Finite sets

Sets are values, ascribable to a power-set category: `𝒫(ℤ)` and `2^ℤ`
are the same collection read two ways. The four binary operations —
union, intersection, difference, symmetric difference — are exact.

In [ ]:
let A := {1, 2, 3} in 𝒫(ℤ)
let B := {3, 4, 5} in 2^ℤ

In [ ]:
assert A ∪ B = {1, 2, 3, 4, 5}
assert A ∩ B = {3}
assert A \ B = {1, 2}
assert A △ B = {1, 2, 4, 5}

In [ ]:
assert |A| = 3
assert |A × B| = 9
assert |𝒫(A)| = 2^|A|

Membership and inclusion are propositions: `∈`, `∉`, `⊆`. The
power-set ascription is **checked** — a set containing a non-integer
element would be refused in `𝒫(ℤ)`.

In [ ]:
assert 2 ∈ A
assert 4 ∉ A
assert A ⊆ A ∪ B
assert A ∩ B ⊆ A

Comprehension: `{n ∈ ℤ | n² ≤ 20}` is the subset of ℤ decided by the
guard. An infinite comprehension like `{2n | n ∈ ℕ}` is not a crash —
it is its progression, and membership is decidable.

In [ ]:
let S := {n ∈ ℤ | n² ≤ 20}
assert S = {-4, -3, -2, -1, 0, 1, 2, 3, 4}
assert |S| = 9

In [ ]:
let m2: ℕ → ℕ := n ↦ 2n
let E := {2n | n ∈ ℕ}
assert 8 ∈ E
assert |E| = ℵ₀

In [ ]:
assert m2(ℕ) = E
assert m2.image() = E

A bounded image comprehension enumerates exactly — the set is the image
of the finite range, no more, no less:

In [ ]:
{m2(n) | n ∈ ℕ, 0 ≤ n < 6}

## 8 · Functions

A function is an expression, not a table: `t ↦ t² + 1` denotes the map
itself, and two functions are equal when they are the same expression —
equality is not sampled at points.

In [ ]:
let h := t ↦ t² + 1 in ℝ → ℝ
let hp(t) := t^2 + 1 in R->R
assert h = hp

In [ ]:
assert h(0) = 1
assert h(3) = 10

Parity is an identity of expressions, seen without any point check:

In [ ]:
assert h(-t) = h(t)

Composition is an identity of function expressions too:

In [ ]:
let sq(t) = t^2 in RR->RR
let cub(t) = t^3 in RR->RR
assert (sq ∘ cub)(t) = t^6

A body the polynomial engine cannot express is SYMBOLIC: a presentation, not a point-evaluable function. Nothing here approximates — the refusal says so. `e` is Euler's constant in these bodies, and `sin`, `1/t` are vocabulary the polynomial reading cannot reach.

In [ ]:
let expo := t ↦ e^t in ℝ → ℝ

In [ ]:
let sine: ℝ → ℝ := t ↦ sin(t)

In [ ]:
let recip := t ↦ 1/t in ℝ → ℝ

In [ ]:
sine(0)

In [ ]:
recip(2)

A lambda needs its domain: the ascription is not optional.

In [ ]:
let bad := t ↦ t^2

Composition is refused when the domains do not compose — `h : ℝ → ℝ` after `m2 : ℕ → ℕ` has no common meeting set — and an argument outside the source domain refuses by name:

In [ ]:
assert (h ∘ m2)(t) = t

In [ ]:
m2(-1)

## 9 · Subspaces and spans

`span_QQ{u₁, u₂}` is the ℚ-span of two vectors, ascribed as a
**subobject**: `\leq ℚ³ in Mod(QQ)` — the `≤` is subobject-in-a-
category, not the numeric relation. The span answers dimension and
membership.

In [ ]:
let u₁ := (1, 0, 1) in ℚ³
let u₂ := (0, 1, 1) in ℚ³
let W := span_QQ{u₁, u₂} \leq ℚ³ in Mod(QQ)

In [ ]:
assert W.dim() = 2
assert (1, 1, 2) ∈ W
assert (1, 1, 0) ∉ W

The same subspace read as a kernel: φ vanishes exactly on `W`. A linear
map is a hom in the category, and its kernel is a subobject again.

In [ ]:
let φ: ℚ³ → ℚ := (a, b, c) ↦ a + b - c
assert W = ker φ

In [ ]:
assert φ((1, 1, 2)) = 0
assert φ((1, 1, 0)) = 2

Homs are first-class, but a hom is not an object of the category it runs in: ascribing the map itself to `Mod(QQ)` is a morphism-is-not-an-object hold, refused by name.

In [ ]:
let gm: QQ^3 -> QQ := (a, b, c) |-> a + b - c in Mod(QQ)

A vector-valued hom is called, composed, and read by `ker` and `im` — the subobject presentations route to the same span machinery spans use.

In [ ]:
let fh: ℚ³ → ℚ³ := (x, y, z) ↦ (2x + 3y + z, x - y, 3z - x)

In [ ]:
assert fh((1, 0, 0)) = (2, 1, -1)

In [ ]:
assert (fh ∘ fh)((1, 0, 0)) = (6, 1, -5)

In [ ]:
assert ker fh = {0}

In [ ]:
assert im fh = span_QQ{(1, 0, 0), (0, 1, 0), (0, 0, 1)}

## 10 · Elementary calculus

Limits are exact values where they exist — evaluated, not approximated.
A limit the backend cannot name is a refusal (§13), never a guess.

In [ ]:
assert lim_{t → 0} sin(t)/t = 1
assert lim_{t → ∞} 1/t = 0

Definite integrals are exact:

In [ ]:
assert ∫₀¹ t² dt = 1/3
assert ∫₀^π sin(t) dt = 2

Taylor expansion is a function into the ring of formal power series —
the jet of $f$ at a point. Truncation `ℝ[[t]]/(t^n)` is a quotient of
that ring, and the map is exact:

In [ ]:
let expf := t ↦ exp(t) in ℝ → ℝ
let Tf := expf.taylor_expansion(0) in ℝ[[t]]
assert Tf ∈ ℝ[[t]]
map Tf to ℝ[[t]]/(t^6)

In [ ]:
let g: ℝ → ℝ := t ↦ sin(t)
let Tg := g.taylor_expansion(0) in ℝ[[t]]
map Tg to ℝ[[t]]/(t^8)

## 11 · A composed computation

Everything composes. The roots of $r$ over ℂ, their elementary
symmetric functions, and the companion matrix all answer from the same
polynomial:

In [ ]:
let r(x) := x³ - 2x + 1 in ℚ[x]
let roots := {a ∈ ℂ | r(a) = 0} in 𝒫(ℂ)
assert |roots| = 3
assert 1 ∈ roots
assert ∑_{a ∈ roots} a = 0
assert ∏_{a ∈ roots} a = -1

In [ ]:
let C := r.companion_matrix()
assert C.charpoly() = r
assert C.det() = -1
assert C.trace() = 0

## 12 · Ellipses

`...` denotes an infinite sequence inferred from a finite pattern:

In [ ]:
let X := {0, 1, 2, ...}
assert X = ℕ
let Y := {0, 2, 4, ...}
assert Y = 2ℕ
assert 8 ∈ Y
assert 9 ∉ Y

The alias layer: the backslash family and the ident aliases are accepted wherever the unicode form goes — `\NN`, `\in`, `\leq` are the same tokens as ℕ, ∈, ≤ — and SPEC's series binding uses the ASCII spellings throughout.

In [ ]:
let Xa := {0, 1, 2, ...}

In [ ]:
assert Xa = \NN

In [ ]:
assert 7/3 \in ℚ

In [ ]:
let fa: NN -> NN := n ↦ n^2

In [ ]:
assert fa(3) = 9

In [ ]:
let fs(t) = ∑_{n ∈ \NN} n^2 t^n \in ZZ[[t]]

In [ ]:
assert [t^2]fs = 4

A guarded comprehension may be lazy: `primes` is the set of primes,
decided on membership — never enumerated.

In [ ]:
let primes := {n in ℕ | n.is_prime()}
assert 13 ∈ primes
assert 15 ∉ primes

Ellipses also fill a finite pattern: `CC[x_0, x_1, ..., x_9]` is the
polynomial ring in ten indeterminates.

In [ ]:
let Rf := CC[x_0, x_1, ..., x_9]

A generating series is an element of the ring of formal power series —
coefficients extractable by `[t^k]`:

In [ ]:
let sq(t) = ∑_{n ∈ ℕ} n^2 t^n ∈ ℤ[[t]]
assert [t^2]sq = 4
assert sq ∈ ℤ[[t]]/(t^5)
map sq to ℤ[[t]] / O(t^5)

## 13 · The audit surface

The diagnostics are part of the surface: `#capabilities` lists what each backend can do; `#capability_gaps` lists what is refused — `det` over ℤ/5, the ceiling below; `#canonical_maps` lists the registered identifications that drive `⊆`, `∈`, `map`, and coercion.

In [ ]:
#capabilities

In [ ]:
#capability_gaps

In [ ]:
#canonical_maps

## 7 · Documented ceiling

Not every mathematically meaningful operation has an implementation yet.
Ask for one and the system refuses, naming the operation, the object it
was asked on, and why no backend can answer — an explicit gap, not a
crash and not a hidden method.

Here `det` is known on Mat₂(ℤ/5) — the category declares the method —
but no backend provides it for matrices over ℤ/5 yet. Over ℚ it works
(we just used it); over ℤ/5 it is a gap.


In [ ]:
let N := [1, 2; 3, 4] in Mat₂(ℤ/5)

In [ ]:
N.det()